## 数据模型

## Schema

- Physical Schema : 描述数据在磁盘上的存储
- Conceptual Schema : 关系数据库间表之间的逻辑模式 (基表)
- View : 用户视图

## 数据独立性

 三级数据模式 + 两级映射 => 两级独立性 
 
 - 逻辑数据独立性 : 逻辑结构变化不影响视图 
 - 物理数据独立性 : 物理结构变化不影响逻辑

> 使用数据库的最主要的优势!!!

## 关系型数据库的理论基础

- 每一个`属性`(attribute)对应的取值范围称为`域`(domin)
- 每一个属性要满足两个条件:(1)原子性(2)允许某些属性值为空
- 现实世界中的实体,或者实体之间的联系会被表示为`关系`
- 一个关系`R`实际上就是一个定义在所有属性取值范围上的N元联系
- $R\ with\ A_1,A_2,...,A_n$每一个属性对应的取值范围(域)$D_1,D_2,...,D_n$
- 一个关系可以被形式化的表示为$R\ =( A_1/D_1,A_2/D_2,...,A_n/D_n)$或者$R\ =( A_1,A_2,...,A_n)$
- 关系的实例组用$r表示$,其中$r = \{t_1,t_2,...,t_m\},t = <v_1,v_2,...,v_n>,v_i\in D_i$
- $t\in D_1\times D_2\times...\times D_n$(笛卡尔积)
- 关系叫`表`,属性叫`域`,每一个实体元组叫`行`
- 候选键(ck,candidate key): (1)一个或者一组属性能够唯一确定一行;(2)这组属性的任何子集没有上述特性
- 超键(sk,superkey): 满足候选键条件1,但是不满足条件2
- 主键(pk,primary key): 候选键可能由有多组,主键属于其中的唯一一组
- 全键(ak,all key): 主键由关系的所有属性确定
- 外键(fk,foreign key): 能够引用另外关系的一组键(在那组关系中应该是主键,在本表中,**外键不允许null**)
- 域完整性约束: 表中的每一个元组每一个属性的值满足约束
- 实体完整性约束:**关系主键不允许null**

实例: `水手sailors`,`船只boats`,`预定reserves`

In [1]:
import pymysql
conn = pymysql.connect(host='127.0.0.1',user='root',password='')
cursor = conn.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS SAILORANDBOAT;")
cursor.execute("USE SAILORANDBOAT")

0

水手订购船信息 R1

|  sid|  bid|  day|
|---|---|---|
| 22 | 101 | 10/10/96 |
| 58 | 103 | 11/12/96 |

In [2]:
cursor.execute("CREATE TABLE R1(sid INT(11),bid INT(11),day date);")

0

In [3]:
sql = "INSERT INTO R1(sid,bid,day) VALUES(%d,%d,'%s')"
cursor.execute(sql%(22,101,'1996/10/10'))
cursor.execute(sql%(58,103,'1996/11/12'))

1

In [4]:
cursor.execute("select * from R1")
cursor.fetchall()

((22, 101, datetime.date(1996, 10, 10)),
 (58, 103, datetime.date(1996, 11, 12)))

船信息 B1

|  bid|  bname|  color|
|---|---|---|
| 101 | tiger | red |
| 102 | lion  | green |
| 103 | hero  | blue |


In [5]:
cursor.execute("CREATE TABLE B1(bid INT(11),bname varchar(15),color varchar(15));")
sql = "INSERT INTO B1(bid,bname,color) VALUES(%d,'%s','%s')"
cursor.execute(sql%(101,'tiger','red'))
cursor.execute(sql%(102,'lion','green'))
cursor.execute(sql%(103,'lhero','blue'))
cursor.execute("select * from B1")
cursor.fetchall()

((101, 'tiger', 'red'), (102, 'lion', 'green'), (103, 'lhero', 'blue'))

水手信息 S1

|  sid|  sname|  rating|age|
|---|---|---|---|
| 22 | dustin | 7 | 45.0 |
| 31 | lubber  | 8 | 55.5 | 
| 58 | rusty  | 10 | 35.0 |

水手信息 S2

|  sid|  sname|  rating|age|
|---|---|---|---|
| 28 | yuppy | 9 | 35.0 |
| 31 | lubber  | 8 | 55.5 | 
| 44 | guppy  | 5 | 35.0 |
| 58 | rusty  | 10 | 35.0 |

In [6]:
cursor.execute("CREATE TABLE S1(sid INT(11),sname varchar(15),rating INT(11),age FLOAT(5,2));")
sql = "INSERT INTO S1(sid,sname,rating,age) VALUES(%d,'%s',%d,%f)"
cursor.execute(sql%(22,'dustin',7,45.0))
cursor.execute(sql%(31,'lubber',8,55.5))
cursor.execute(sql%(58,'rusty',10,35.0))
cursor.execute("select * from S1")
print(cursor.fetchall())

cursor.execute("CREATE TABLE S2(sid INT(11),sname varchar(15),rating INT(11),age FLOAT(5,2));")
sql = "INSERT INTO S2(sid,sname,rating,age) VALUES(%d,'%s',%d,%f)"
cursor.execute(sql%(28,'yuppy',9,35.0))
cursor.execute(sql%(31,'lubber',8,55.5))
cursor.execute(sql%(44,'guppy',5,35.0))
cursor.execute(sql%(58,'rusty',10,35.0))
cursor.execute("select * from S2")
print(cursor.fetchall())

((22, 'dustin', 7, 45.0), (31, 'lubber', 8, 55.5), (58, 'rusty', 10, 35.0))
((28, 'yuppy', 9, 35.0), (31, 'lubber', 8, 55.5), (44, 'guppy', 5, 35.0), (58, 'rusty', 10, 35.0))


## 关系代数基础

只要一个系统支持下述5个操作,即关系完备

- Selection($\sigma$): 从关系中选择行子集
- Projection($\pi$): 去除不需要的列,留下需要的列,**投影操作会消除重复元素**,但是真实的数据库产品不会进行这样的操作
- Cross-product($\Join$): 笛卡尔积
- Set-difference($-$): 集合差
- Union($\cup$): 集合并


### 投影操作结果关系的模式

In [9]:
cursor.execute("select age from S2")
print(cursor.fetchall())
cursor.execute("select sname,age age from S2")
print(cursor.fetchall())

((35.0,), (55.5,), (35.0,), (35.0,))
(('yuppy', 35.0), ('lubber', 55.5), ('guppy', 35.0), ('rusty', 35.0))


### 选择操作

选出表中满足条件的行

$\sigma_{rating>8}S2$

In [11]:
cursor.execute("select * from S2 where rating > 8;")
cursor.fetchall()

((28, 'yuppy', 9, 35.0), (58, 'rusty', 10, 35.0))

操作的组合: 

选择&投影
$\pi_{sname.rating}(\sigma_{rating>8}(S2))$

水手信息 S2

|  sid|  sname|  rating|age|
|---|---|---|---|
| 28 | yuppy | 9 | 35.0 |
| 31 | lubber  | 8 | 55.5 | 
| 44 | guppy  | 5 | 35.0 |
| 58 | rusty  | 10 | 35.0 |

In [12]:
cursor.execute("select sname,rating from S2 where rating > 8;")
cursor.fetchall()

(('yuppy', 9), ('rusty', 10))

## 集合的`并交叉`

要求:**参与集合允许的关系满足并兼容**
- 两个关系有相同个数的属性
- 对应属性类型一致
  
并 $S1\cup S2$  
交 $S1\cap S2$  
叉 $S1-S2$  

水手信息 S1

|  sid|  sname|  rating|age|
|---|---|---|---|
| 22 | dustin | 7 | 45.0 |
| 31 | lubber  | 8 | 55.5 | 
| 58 | rusty  | 10 | 35.0 |


水手信息 S2

|  sid|  sname|  rating|age|
|---|---|---|---|
| 28 | yuppy | 9 | 35.0 |
| 31 | lubber  | 8 | 55.5 | 
| 44 | guppy  | 5 | 35.0 |
| 58 | rusty  | 10 | 35.0 |

In [29]:
cursor.execute("select * from S2 union select * from S1;")
print("并",cursor.fetchall())
cursor.execute("select S2.sid,S2.sname,S2.rating,S2.age from S2 INNER JOIN S1 on S1.sid = S2.sid;")
print("交",cursor.fetchall())
cursor.execute("""
select * from S2 WHERE NOT EXISTS (select * from S1 where S1.sid = S2.sid );
""") # todo - not in 的效率并不好 该用exists
print("叉",cursor.fetchall())

并 ((28, 'yuppy', 9, 35.0), (31, 'lubber', 8, 55.5), (44, 'guppy', 5, 35.0), (58, 'rusty', 10, 35.0), (22, 'dustin', 7, 45.0))
交 ((31, 'lubber', 8, 55.5), (58, 'rusty', 10, 35.0))
叉 ((28, 'yuppy', 9, 35.0), (44, 'guppy', 5, 35.0))


### 笛卡尔乘积-连接操作 Join

$$
R\Join_C S = \sigma_C(R\times S)
$$

- 条件连接

$S1\Join_{S1.sid < R1.sid}R1$

- 等值连接

$S1\Join_{sid}R1$

In [36]:
sql = "select * from S1 INNER JOIN R1 on  S1.sid < R1.sid;"
cursor.execute(sql)
print("条件连接",cursor.fetchall())

sql = "select * from S1 INNER JOIN R1 on  S1.sid = R1.sid;"
cursor.execute(sql)
print("等值连接",cursor.fetchall())

条件连接 ((22, 'dustin', 7, 45.0, 58, 103, datetime.date(1996, 11, 12)), (31, 'lubber', 8, 55.5, 58, 103, datetime.date(1996, 11, 12)))
等值连接 ((22, 'dustin', 7, 45.0, 22, 101, datetime.date(1996, 10, 10)), (58, 'rusty', 10, 35.0, 58, 103, datetime.date(1996, 11, 12)))


### 外连接

- left outer join ($*\Join$)
- right outer join ($\Join*$)
- full outer join ($*\Join*$)

### 外并
- outer union 不满足并兼容情况下


### 除法操作

$A/B=\{<x>|\exists <x,y> \in A\ , \forall <y> \in B)\}$

> 水手订过所有的船

A/B 在关系A中找到与关系B中所有y值都有联系的x值

=>找到这样的x: 没有一个y值与x是没有联系的

$A/B =\pi_x (A)- \pi_x((\pi_x(A)\times B)-A)$


In [53]:
# todo 
sql = """
select S1.sid,B1.bid from S1 JOIN  B1
"""
cursor.execute(sql)
cursor.fetchall()

((22, 101),
 (31, 101),
 (58, 101),
 (22, 102),
 (31, 102),
 (58, 102),
 (22, 103),
 (31, 103),
 (58, 103))

In [15]:
cursor.close()
conn.close()

## 关系演算

- TRC 元组关系演算
- DRC 域关系演算

定义:

- $Query:\{<x_1,x_2,...,x_n> | P(x1,x_2,...,x_n,x_{n+1},...x_{n+m}) \}$
- $x1,x_2,...,x_n,x_{n+1},...x_{n+m}$ 称为域
- 找到一组属性值集合能够满足条件的实体

原子公式:,
- $<x_1,x_2,...,x_n>\in Rname$,$X\ op\ Y$,$X\ op\ constant$,其中$op\in \{\neq,=,<,>,\ge,\le\}$

公式
- 原子公式
- $\neg p,p\wedge q,p\vee q$
- 存在量词:$\exists X(p(x))$
- 全称量词:$\forall X(p(x))$

自由变量和绑定变量
- 量词描述的是绑定变量$\exists X,\forall X$
- 自由变量查询,左边的

> 例子:查找级别大于7的水手

> 关系代数:$\sigma_{rating>7}S1$  

> 关系演算:$\{<I,N,T,A>|<I,N,T,A>\in Sailors\ \wedge T>7 \}$

安全查询: 结果是一个确定的有限结合

元组关系演算:
- $\{t[<attr list>]|P(t)\}$
- $\{ t[N] | t\in Sailors \wedge t.T>7 \wedge t.A<50 \}$

## E-R数据模型

- 实体 : 现实世界中的可以区分的真实对,实体由一组属性描述 
- 实体集: 一组类似实体的集合
    - 一个实体集中所有的实体有相同的属性
    - 每一个实体有一个key
    - 每一个属性有一个domin
    - 允许复合类型,多值属性
- 联系: 实体之间的关系
- 联系集: 所有同类联系的集合
    - 组成联系的n个实体的n元关系
    - 同一个实体集可以参与不同的联系
 
联系可以表示为:
- 1 to 1
- 1 to m
- m to 1
- m to m

ER模型中的高级概念:
- 弱实体
- 泛化和特殊化
- 聚合: 可以把联系也看成实体
- 范畴: 不同类型实体的集合